### **Bert Fine-tuning**
This is the first notebook for Task 4, downloaded directly from Google Colab. It is exclusively dedicated to fine-tuning the pre-trained Italian model (`dbmdz/bert-base-italian-cased`) to detect machine-generated text. 

Because training deep learning models requires a lot of computational power, this specific step was separated from the rest of the project and run in the cloud using a T4 GPU.

In [ ]:
import pandas as pd
import numpy as np
import torch
import pandas as pd
from google.colab import drive
import os

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

# Ensure GPU is used if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### Environment Setup and Data Loading
This cell mounts Google Drive to access the project's persistent storage. It then loads the pre-processed datasets (Training, Validation, and Test splits) into pandas DataFrames and verifies their dimensions.

In [ ]:
# 1. Mount Google Drive to access your files
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Load the dataframes from the 'lc2' folder in My Drive
print("Loading dataframes from 'lc2' folder...")
df_train = pd.read_pickle("/content/drive/MyDrive/lc2/train_final.pkl")
df_val = pd.read_pickle("/content/drive/MyDrive/lc2/val_final.pkl")
df_test = pd.read_pickle("/content/drive/MyDrive/lc2/test_final.pkl")

# 3. Verify dataset dimensions
print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataframes from 'lc2' folder...
Train shape: (4000, 147)
Validation shape: (2000, 147)
Test shape: (2000, 147)


### Dataset Conversion and Tokenization
This cell isolates the required textual and label data, converting the pandas DataFrames into Hugging Face `Dataset` objects. It initializes the pre-trained Italian BERT tokenizer (`dbmdz/bert-base-italian-cased`) and applies a batched tokenization function that ensures uniform sequence lengths via padding and truncation (up to 512 tokens). Finally, the datasets are formatted as PyTorch tensors to enable seamless integration with the subsequent neural network training loop.

In [ ]:
# Convert to Hugging Face Datasets
# Only the 'text' and 'label' columns are required for the Neural Language Model
train_dataset = Dataset.from_pandas(df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text', 'label']])
test_dataset = Dataset.from_pandas(df_test[['text', 'label']])

# Load the pre-trained Tokenizer
model_checkpoint = "dbmdz/bert-base-italian-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # Truncate to 512 tokens (BERT's maximum context length) and pad
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

Tokenizing datasets...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

### Model Initialization and Fine-Tuning
This cell loads the pre-trained Italian BERT model configured for binary sequence classification. It establishes a custom evaluation metric to track the Macro-F1 score and defines the hyperparameter configuration, including learning rate, batch size, and a strict 3-epoch training limit. Finally, the Hugging Face `Trainer` is instantiated and executed to fine-tune the network on the training set, evaluating against the validation set to automatically retain the best-performing model weights.

In [ ]:
# Load the pre-trained sequence classification model
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
model.to(device)

def compute_metrics(eval_pred):
    """
    Custom metric function to compute Macro-F1 during evaluation.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"macro_f1": macro_f1}

# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",          # Evaluate at the end of each epoch
    save_strategy="epoch",          # Save checkpoint at the end of each epoch
    logging_strategy="epoch",       # Log metrics at the end of each epoch
    learning_rate=2e-5,             # Standard learning rate for BERT fine-tuning
    per_device_train_batch_size=8,  # Adjust based on GPU VRAM (16 is also good if it fits)
    per_device_eval_batch_size=8,
    num_train_epochs=3,             # Exactly 3 epochs as required
    load_best_model_at_end=True,    # Automatically restore the best checkpoint
    metric_for_best_model="macro_f1",
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("--- Starting Fine-Tuning ---")
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-italian-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Starting Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Macro F1
1,0.122897,0.042244,0.991000
2,0.022561,0.021258,0.996500
3,0.004960,0.027338,0.996000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=0.05013936853408813, metrics={'train_runtime': 1346.6551, 'train_samples_per_second': 8.911, 'train_steps_per_second': 1.114, 'total_flos': 3157332664320000.0, 'train_loss': 0.05013936853408813, 'epoch': 3.0})

### Model Export
This cell exports the fine-tuned model weights alongside its associated tokenizer to the persistent Google Drive storage. This ensures the optimized artifacts are securely saved and can be readily reloaded for future inference or evaluation without the need to repeat the training phase.

In [ ]:
# Define output path on your Google Drive
save_directory = "/content/drive/MyDrive/lc2"

# Save the trained model and tokenizer
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model successfully saved to {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to /content/drive/MyDrive/lc2
